# Optimization

**OBJECTIVES**

- Define and plot functions using `numpy` and `sympy`
- Find derivatives of functions approximately and exactly using Python
- Determine minimum values of functions using derivatives
- Approximate minimum values of functions using gradient descent

In [ ]:
import sympy as sy
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import numpy as np

## Basic Functions

Consider the function

$$f(x) = x^2 - 2x + 3$$

below the function is defined and plotted.

In [ ]:
#define the function
def f(x):
    return x**2 - 2*x + 3

In [ ]:
#define domain
x = np.linspace(-2, 4, 100)

In [ ]:
#plot on domain
plt.plot(x, f(x));

From the plot above can you approximate where $f$ takes a minimum value?

### Calculus Refresher

Recall that a function achieves a minimum value where the first derivative equals zero.

$$f'(x) = 0$$

The `SymPy` library allows us to solve problems like this symbolically on the computer.  In order to work with symbols, we have to explicitly define them.

Further, we have our convexity to check with the second derivative.  For a minimum we wanted concave up aka convex:

$$f''(x_0) > 0$$

In [ ]:
x, y = sy.symbols('x y')

In [ ]:
sy.diff(x**2 - 2*x + 3, x)

In [ ]:
equation = sy.Eq(2*x - 2, 0)

In [ ]:
solution = sy.solve(equation, x)

In [ ]:
print(solution)

In [ ]:
sy.diff(x**2 - 2*x + 3, x, 2)

**PROBLEM**

1. Show that the function $$f(x) = x^4 - 4x^3 - 2x^2 + 12x$$ takes on a minimum value at $x = -1$ and $x = 3$.  Show that it takes on a maximum value at $x = 1$.  Plot the function and these points on the graph.  Which of the points are local optima?


2. Find the local and global optima of $f(x) = (x - 1)^2[(x-3)^2 - 1]$. Which of these are maxima and which are minima?

## Numerically Approximating the Minimum

A different approach to finding minimum values is to use tangent lines (or planes) to guide us to the minimum value.  Consider the plot below of the function $$f(x) = x^2$$ and the line tangent to the curve at $x = 2$.

In [ ]:
def f(x): return x**2
def df(x): return 2*x
def tan_line(a): return df(a)*(x - a) + f(a)
x = np.linspace(-3, 3, 100)
plt.plot(x, f(x))
plt.plot(x, tan_line(2), '-r')
plt.plot(2, f(2), 'ro')
plt.ylim(-1, 10)
plt.grid();

**QUESTION**

- If you were to start at $x = 2$ and step one unit to the right ($\alpha = 1$), where would you end up in terms of a $y$ value if you follow the tangent line?
- What if you were to step $\frac{1}{4}$ a unit to the right using the tangent line as a guide?  Where will you end up?
- What happens if you step of $\frac{1}{10}$ in the opposite direction of the slope of the tangent line? What is your new $x-$coordinate.

**PROBLEM**

Can you explain why this formula works to answer the last question?

$$x_{i+1} = x_i - \alpha f'(x_i)$$

where $x_{i + 1}$ represents the next $x$-value and $x_i$ the previous.

### Gradient Descent

Now we have a method to find the minimum of functions approximately.  We should revisit the problem of $f(x) = x^2$ and starting at $x = 2$ we want to perform gradient descent and update the approximations for the minimum value iteratively.

1. Perform 5 iterations of gradient descent starting at $x = 2$.  Where do you end up?

2. Write a loop to keep track of your updated approximations and plot the resulting descent using 50 iterations.

#### Using the chain rule

**Example 1**: Line of best fit

In [ ]:
import seaborn as sns

In [ ]:
tips = sns.load_dataset('tips')

In [ ]:
tips.head()

In [ ]:
sns.scatterplot(data = tips, x = 'total_bill', y = 'tip');

In [ ]:
def y1(x): return .19*x

In [ ]:
def y2(x): return .12*x

In [ ]:
x = tips['total_bill']

In [ ]:
plt.plot(x, y1(x), label = 'y1')
plt.plot(x, y2(x), label = 'y2')
plt.legend()
sns.scatterplot(data = tips, x = 'total_bill', y = 'tip')
plt.title('Which line is a better fit?')
plt.grid();

To decide between all possible lines we will examine the error in all these models and select the one that minimizes this error.

In [ ]:
plt.plot(x, y2(x))
for i, yhat in enumerate(y2(x)):
    plt.vlines(x = tips['total_bill'].iloc[i], ymin = yhat, ymax = tips['tip'].iloc[i], color = 'red', linestyle = '--')
sns.scatterplot(data = tips, x = 'total_bill', y = 'tip')
plt.title('Error in Model')
plt.grid();

#### Mean Squared Error

$$\text{MSE}(\beta_0) = \frac{1}{n}\sum_{i = 1}^n (y_i - \beta_0x)^2$$



**OBJECTIVE**: Minimize mean squared error

In [ ]:
def mse(beta):
    return np.mean((y - beta*x)**2)

In [ ]:
x = tips['total_bill']
y = tips['tip']

In [ ]:
mse(.17)

In [ ]:
for pct in np.linspace(.1, .2, 11):
    print(f'The MSE for slope {pct: .3f} is {mse(pct): .3f}')

In [ ]:
betas = np.linspace(0, .4, 100)
plt.plot(betas, [mse(beta) for beta in betas])
plt.xlabel(r'$\beta$')
plt.ylabel('MSE')
plt.title('Mean Squared Error for Different Slopes')
plt.grid();

To perform gradient descent, we need the gradient of the MSE function.  Determine this gradient and write a loop to update the slope with minimum MSE starting at $x = .25$.

#### Using `scipy`

To find the minimum of our objective function, the `minimize` function from `scipy.optimize` is useful.  This relies on a variety of different optimization algorithms to find the minimum of a function.

In [ ]:
from scipy.optimize import minimize

In [ ]:
minimize(mse, .1 )

**PROBLEM**

Return to our earlier examples.  Use `scipy` to verify your earlier results.

### Solving the Problem Exactly

From calculus we know that the minimum value for a quadratic will occur where the first derivative equals zero.  Thus, to determine the equations for the line of best fit, we minimize the MSE function with respect to $\beta$.

$$f(\beta) = \frac{1}{n}\sum_{i = 1}^n (y - \beta x)^2$$

$$f'(\beta) = \frac{-2}{n}\sum_{i = 1}^n(y - \beta x) x$$

$$ 0 = \frac{-2}{n}\sum_{i = 1}^n(y - \beta x) x$$

$$0 = \sum_{i = 1}^n(y - \beta x) x$$

$$0 = \sum yx - \sum \beta x^2 $$

$$\sum \beta x^2 = \sum y x$$

$$\beta \sum x^2 = \sum y x$$

$$\beta = \frac{\sum y x}{\sum x^2}$$

In [ ]:
np.sum(tips['total_bill']*tips['tip'])/np.sum(tips['total_bill']**2)

### Problem: Determine $\alpha$ and $\beta$ of an investment

Consider an investment with return of the market as $R_M$ (say the S&P) and the return of a stock $R$ over the same time period.

$$R = \alpha + \beta*R_M + \epsilon$$

In [ ]:
import yfinance as yf

In [ ]:
data = yf.download(["NVDA", "SPY"], period="5y")
data.head()

In [ ]:
data.info()

In [ ]:
close = data.iloc[:, [0, 1]]

In [ ]:
close.pct_change().plot()

In [ ]:
returns = close.pct_change()

In [ ]:
sns.regplot(x = returns.iloc[:, 1], y = returns.iloc[:, 0], scatter_kws={'alpha': 0.3})
plt.xlabel('SPY Close Pct Change')
plt.ylabel('NVDA Close Pct Change')
plt.grid();

In [ ]:
returns.head(2)

In [ ]:
#instantiate
returns.dropna(inplace = True)
X = returns.iloc[:, [1]]
y = returns.iloc[:, [0]]
model = LinearRegression()

In [ ]:
#fit
model.fit(X, y)

In [ ]:
#examine our alpha and beta
alpha = model.intercept_
beta = model.coef_
print(alpha, beta)

**EXIT TICKET**: [here](https://docs.google.com/forms/d/e/1FAIpQLSdMfZqZTiJg81aIIOnLt_Ye6SeOfZWrcorxcqXl5tyHLdvPTA/viewform?usp=sharing&ouid=106596295392954890759)